# Common preprocessing for diabetic dataset

Notebook này tạo một bản dữ liệu sạch chung sau EDA. File output sẽ được dùng làm đầu vào cho 3 hướng tiếp theo: phân loại, phân cụm và luật kết hợp.

Mục tiêu của bước chung:

- Đọc dữ liệu gốc `diabetic_data.csv`.
- Chuẩn hóa missing value từ `?` thành `NaN`.
- Loại bỏ các cột định danh hoặc gần như không có thông tin mô hình hóa.
- Xử lý một số giá trị bất thường/còn thiếu ở mức nền tảng.
- Tạo các feature dùng chung như `readmitted_binary`, nhóm tuổi dạng số, nhóm ICD-9.
- Lưu dữ liệu sạch ra `data/diabetic_data_clean_common.csv`.


In [13]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)


In [14]:
cwd = Path.cwd().resolve()
ROOT_DIR = cwd if (cwd / "diabetic_data.csv").exists() else cwd.parent
RAW_DATA_PATH = ROOT_DIR / "diabetic_data.csv"
OUTPUT_DIR = ROOT_DIR / "data"
OUTPUT_PATH = OUTPUT_DIR / "diabetic_data_clean_common.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH, OUTPUT_PATH


(WindowsPath('D:/HocTap/KT&XLTT/CUOIKI/diabetic_data.csv'),
 WindowsPath('D:/HocTap/KT&XLTT/CUOIKI/data/diabetic_data_clean_common.csv'))

## 1. Load dữ liệu gốc


In [15]:
df_raw = pd.read_csv(RAW_DATA_PATH)
df = df_raw.copy()

print("Raw shape:", df.shape)

df.tail(5)


Raw shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
101761,443847548,100162476,AfricanAmerican,Male,[70-80),?,1,3,7,3,MC,?,51,0,16,0,0,0,250.13,291,458,9,NaN,>8,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,>30
101762,443847782,74694222,AfricanAmerican,Female,[80-90),?,1,4,5,5,MC,?,33,3,18,0,0,1,560,276,787,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,NO
101763,443854148,41088789,Caucasian,Male,[70-80),?,1,1,7,1,MC,?,53,0,9,1,0,0,38,590,296,13,NaN,NaN,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,NO
101764,443857166,31693671,Caucasian,Female,[80-90),?,2,3,7,10,MC,Surgery-General,45,2,21,0,0,1,996,285,998,9,NaN,NaN,No,No,No,No,No,No,Steady,No,No,Steady,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
101765,443867222,175429310,Caucasian,Male,[70-80),?,1,1,7,6,?,?,13,3,3,0,0,0,530,530,787,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO


## 2. Chuẩn hóa missing value

Trong dataset này, nhiều giá trị thiếu được ghi bằng dấu `?`, nên cần đổi sang `NaN` để xử lý nhất quán.


In [16]:
df = df.replace("?", np.nan)

missing_summary = (
    df.isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_rate=lambda x: x["missing_count"] / len(df))
    .query("missing_count > 0")
    .sort_values("missing_rate", ascending=False)
)

missing_summary


,missing_count,missing_rate
weight,98569,0.968585
max_glu_serum,96420,0.947468
A1Cresult,84748,0.832773
medical_specialty,49949,0.490822
payer_code,40256,0.395574
race,2273,0.022336
diag_3,1423,0.013983
diag_2,358,0.003518
diag_1,21,0.000206


## 3. Bỏ các cột không phù hợp cho bản clean chung

- `encounter_id`, `patient_nbr`: cột định danh, không đưa trực tiếp vào feature.
- `weight`: missing quá cao.
- `examide`, `citoglipton`: thường chỉ có một giá trị, gần như không mang thông tin phân biệt.


In [17]:
columns_to_drop = [
    "encounter_id",
    "patient_nbr",
    "weight",
    "examide",
    "citoglipton",
]

df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

print("Shape after dropping common unused columns:", df.shape)
df.head()


Shape after dropping common unused columns: (101766, 45)


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,Caucasian,Female,[0-10),6,25,1,1,NaN,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,Caucasian,Female,[10-20),1,1,7,3,NaN,NaN,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,NaN,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,Caucasian,Male,[30-40),1,1,7,2,NaN,NaN,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,Caucasian,Male,[40-50),1,1,7,1,NaN,NaN,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


## 4. Xử lý missing và giá trị bất thường mức nền tảng

Ở bước chung, ta chỉ xử lý các giá trị rõ ràng. Các chiến lược encode/scale chi tiết sẽ để riêng cho từng bài toán.


In [18]:
unknown_fill_columns = ["race", "payer_code", "medical_specialty"]

for col in unknown_fill_columns:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

if "gender" in df.columns:
    df = df[df["gender"] != "Unknown/Invalid"].copy()

print("Shape after basic missing handling:", df.shape)
df[unknown_fill_columns + ["gender"]].head()



Shape after basic missing handling: (101763, 45)


,race,payer_code,medical_specialty,gender
0,Caucasian,Unknown,Pediatrics-Endocrinology,Female
1,Caucasian,Unknown,Unknown,Female
2,AfricanAmerican,Unknown,Unknown,Female
3,Caucasian,Unknown,Unknown,Male
4,Caucasian,Unknown,Unknown,Male


## 5. Tạo biến mục tiêu nhị phân

Biến này phục vụ bài toán phân loại nhị phân. Bản clean chung vẫn giữ `readmitted` gốc để nếu cần có thể làm phân loại 3 lớp hoặc dùng trong phân tích luật kết hợp.


In [19]:
if "readmitted" in df.columns:
    df["readmitted_binary"] = np.where(df["readmitted"].eq("NO"), 0, 1)

df[["readmitted", "readmitted_binary"]].head()


,readmitted,readmitted_binary
0,NO,0
1,>30,1
2,NO,0
3,NO,0
4,NO,0


## 6. Tạo biến tuổi dạng số/thứ tự

`age` là nhóm tuổi có thứ tự. Bản clean chung giữ `age` gốc và thêm `age_midpoint`, `age_ordinal` để các bước sau chọn cách dùng phù hợp.


In [20]:
age_order = [
    "[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
    "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)",
]
age_midpoint_map = {
    "[0-10)": 5,
    "[10-20)": 15,
    "[20-30)": 25,
    "[30-40)": 35,
    "[40-50)": 45,
    "[50-60)": 55,
    "[60-70)": 65,
    "[70-80)": 75,
    "[80-90)": 85,
    "[90-100)": 95,
}
age_ordinal_map = {age: idx for idx, age in enumerate(age_order)}

if "age" in df.columns:
    df["age_midpoint"] = df["age"].map(age_midpoint_map)
    df["age_ordinal"] = df["age"].map(age_ordinal_map)

df[["age", "age_midpoint", "age_ordinal"]].head()


,age,age_midpoint,age_ordinal
0,[0-10),5,0
1,[10-20),15,1
2,[20-30),25,2
3,[30-40),35,3
4,[40-50),45,4


## 7. Gom nhóm mã chẩn đoán ICD-9

Các cột `diag_1`, `diag_2`, `diag_3` có rất nhiều mã khác nhau. Bản clean chung thêm các cột nhóm bệnh lớn để giảm số lượng giá trị phân loại.


In [21]:
def map_icd9_group(code):
    if pd.isna(code):
        return "Unknown"

    code_str = str(code).strip()
    if code_str.startswith("V"):
        return "Supplementary_V"
    if code_str.startswith("E"):
        return "Supplementary_E"

    try:
        code_num = float(code_str)
    except ValueError:
        return "Other"

    if 1 <= code_num <= 139:
        return "Infectious_Parasitic"
    if 140 <= code_num <= 239:
        return "Neoplasms"
    if 240 <= code_num <= 279:
        return "Endocrine_Metabolic"
    if 280 <= code_num <= 289:
        return "Blood"
    if 290 <= code_num <= 319:
        return "Mental_Disorders"
    if 320 <= code_num <= 389:
        return "Nervous_Sense_Organs"
    if 390 <= code_num <= 459:
        return "Circulatory"
    if 460 <= code_num <= 519:
        return "Respiratory"
    if 520 <= code_num <= 579:
        return "Digestive"
    if 580 <= code_num <= 629:
        return "Genitourinary"
    if 630 <= code_num <= 679:
        return "Pregnancy_Childbirth"
    if 680 <= code_num <= 709:
        return "Skin"
    if 710 <= code_num <= 739:
        return "Musculoskeletal"
    if 740 <= code_num <= 759:
        return "Congenital"
    if 760 <= code_num <= 779:
        return "Perinatal"
    if 780 <= code_num <= 799:
        return "Symptoms"
    if 800 <= code_num <= 999:
        return "Injury_Poisoning"

    return "Other"


for diag_col in ["diag_1", "diag_2", "diag_3"]:
    if diag_col in df.columns:
        df[f"{diag_col}_group"] = df[diag_col].apply(map_icd9_group)

df[["diag_1", "diag_1_group", "diag_2", "diag_2_group", "diag_3", "diag_3_group"]].head()


,diag_1,diag_1_group,diag_2,diag_2_group,diag_3,diag_3_group
0,250.83,Endocrine_Metabolic,NaN,Unknown,NaN,Unknown
1,276,Endocrine_Metabolic,250.01,Endocrine_Metabolic,255,Endocrine_Metabolic
2,648,Pregnancy_Childbirth,250,Endocrine_Metabolic,V27,Supplementary_V
3,8,Infectious_Parasitic,250.43,Endocrine_Metabolic,403,Circulatory
4,197,Neoplasms,157,Neoplasms,250,Endocrine_Metabolic


## 8. Kiểm tra nhanh dữ liệu sau clean


In [22]:
print("Final shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())

remaining_missing = (
    df.isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_rate=lambda x: x["missing_count"] / len(df))
    .query("missing_count > 0")
    .sort_values("missing_rate", ascending=False)
)

remaining_missing


Final shape: (101763, 51)
Duplicate rows: 0


,missing_count,missing_rate
max_glu_serum,96417,0.947466
A1Cresult,84745,0.832768
diag_3,1423,0.013983
diag_2,358,0.003518
diag_1,21,0.000206


In [23]:
df.head()


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,readmitted_binary,age_midpoint,age_ordinal,diag_1_group,diag_2_group,diag_3_group
0,Caucasian,Female,[0-10),6,25,1,1,Unknown,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO,0,5,0,Endocrine_Metabolic,Unknown,Unknown
1,Caucasian,Female,[10-20),1,1,7,3,Unknown,Unknown,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30,1,15,1,Endocrine_Metabolic,Endocrine_Metabolic,Endocrine_Metabolic
2,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,Unknown,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO,0,25,2,Pregnancy_Childbirth,Endocrine_Metabolic,Supplementary_V
3,Caucasian,Male,[30-40),1,1,7,2,Unknown,Unknown,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO,0,35,3,Infectious_Parasitic,Endocrine_Metabolic,Circulatory
4,Caucasian,Male,[40-50),1,1,7,1,Unknown,Unknown,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO,0,45,4,Neoplasms,Neoplasms,Endocrine_Metabolic


## 9. Lưu dữ liệu clean chung

File này chưa phải là dữ liệu cuối cùng cho mô hình. Từ file này, ta sẽ tạo 3 pipeline riêng:

- Phân loại: encode, split train/test, scale nếu cần.
- Phân cụm: bỏ nhãn mục tiêu, encode, scale mạnh hơn.
- Luật kết hợp: rời rạc hóa biến số và chuyển sang transaction/basket one-hot.
